# Which Types of Monsters Have the Most Hit Points?

**MongoDB Atlas from Colab**

In Dungeons & Dragons, every monster has a type, such as dragon, beast, or undead, and a number of hit points: the
amount of damage the monster takes before it is defeated. This notebook answers the question in the title with data
stored in MongoDB.

MongoDB stores each record as a document, a JSON-like object with named fields. MongoDB groups documents into
collections, the MongoDB counterpart of tables. This notebook connects Python to a MongoDB Atlas cluster. It loads
334 monster descriptions from the Dungeons & Dragons System Reference Document into a collection and queries them. It
then computes the median hit points for each type with pandas and saves the answer back to the cluster as a new
document. Two more parts follow: an index on one field, and a vector search index that finds monsters by the meaning
of a text query.

**How to run the notebook:**

1. Open the notebook in one browser tab only. Two tabs of the same notebook share one Colab runtime, and a password
   box that waits in one tab stops every cell in the other tab.
2. Run the cells in order, from top to bottom. To run a cell, click it and press **Shift+Enter**. Each cell uses
   results from the cells above it.
3. On the first run, Colab shows the warning "This notebook was not authored by Google". Click **Run anyway**.

**Before you start:** the password for the shared class cluster, given in class. For your own Atlas cluster from
the Week 12 Atlas setup page: its database user name, that user's password, and the cluster host.

The notebook writes to a new database, `tabletop`. The `car_market` database from the setup page stays as it is.

**Data source:** [5e-bits/5e-database](https://github.com/5e-bits/5e-database), file
`src/2014/en/5e-SRD-Monsters.json`. This work includes material taken from the System Reference Document 5.1
("SRD 5.1") by Wizards of the Coast LLC, available at
[dnd.wizards.com/resources/systems-reference-document](https://dnd.wizards.com/resources/systems-reference-document).
The SRD 5.1 is licensed under the
[Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/legalcode).

## 1. Install PyMongo and connect

[PyMongo](https://pymongo.readthedocs.io/en/stable/) is MongoDB's Python driver. The first cell installs PyMongo 4.7
or later and [sentence-transformers](https://sbert.net/). PyMongo 4.7 or later can create vector search indexes.
Part 7 uses sentence-transformers to turn text into lists of numbers.

In [ ]:
!pip -q install "pymongo>=4.7" sentence-transformers

The next cell connects to an Atlas cluster. Its form has three fields: `DB_USER` (the database user name),
`CLUSTER_HOST` (the cluster host), and `YOUR_NAME`. The `DB_USER` and `CLUSTER_HOST` defaults connect to the shared
class cluster:

```
mongodb+srv://gsalu2:<db_password>@ncfdbf24.e0wdx.mongodb.net/
```

1. In `YOUR_NAME`, replace `First Last` with your first and last name. Part 5 saves your result under this name.
2. For your own cluster only: in Atlas, click **Connect** on your cluster, then **Drivers**, and copy the connection
   string. In `DB_USER`, put the text between `mongodb+srv://` and the first `:`. In `CLUSTER_HOST`, put the text
   between `@` and the final `/`.
3. Run the cell. A box labeled `Password for gsalu2:` (or your user name) appears under the cell. Type the password
   and press **Enter**.

The box hides the password, so the password does not appear in the notebook.
[`quote_plus`](https://docs.python.org/3/library/urllib.parse.html#urllib.parse.quote_plus) encodes characters such as
`@` and `!` so that the password fits inside the connection string.

`ping` is a command that checks the connection. When it fails, the cause is usually one of two problems: the
**Network Access** list in Atlas does not include `0.0.0.0/0`, or the user name and password belong to the Atlas
account login in place of the database user.

In [ ]:
#@title Connect to Atlas
DB_USER = "gsalu2"  #@param {type:"string"}
CLUSTER_HOST = "ncfdbf24.e0wdx.mongodb.net"  #@param {type:"string"}
YOUR_NAME = "First Last"  #@param {type:"string"}

from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient

password = getpass(f"Password for {DB_USER}: ")
uri = f"mongodb+srv://{DB_USER}:{quote_plus(password)}@{CLUSTER_HOST}/"

client = MongoClient(uri)
client.admin.command("ping")
print("Connected to", CLUSTER_HOST, "as", DB_USER)
print("MongoDB version", client.server_info()["version"])

db = client["tabletop"]
monsters = db["monsters"]

## 2. Load the monsters into a collection

The data is one JSON file: an array of 334 objects, one per monster. `json.load` reads the array into a Python list
of dictionaries. MongoDB creates the `tabletop` database and the `monsters` collection when the first document is
written.

[`bulk_write`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.bulk_write)
sends all 334 writes in one call. Each write is an `UpdateOne` with `upsert=True`, called an upsert: it updates the
document whose `index` field matches, or inserts a new document when nothing matches. `index` is the short name of
the monster in the data, such as `goblin`. The collection keeps one document per monster, however many times the
cell runs and however many notebooks share the cluster.

In [ ]:
import json
import urllib.request
from pymongo import UpdateOne

URL = ("https://raw.githubusercontent.com/5e-bits/5e-database/"
       "a6212beb4b278917c2cff41c2b04c4ef4f3e0d6f/src/2014/en/5e-SRD-Monsters.json")
data = json.load(urllib.request.urlopen(URL))

result = monsters.bulk_write([
    UpdateOne({"index": m["index"]}, {"$set": m}, upsert=True)
    for m in data
])
print(result.upserted_count, "inserted,", result.matched_count, "already present")
print(monsters.count_documents({}), "documents in the collection")

`find_one` returns one document. The second argument is a projection: it lists the fields to return.

In the Adult Red Dragon document, `speed` is an embedded document: a document stored as the value of a field. The fields of `speed` differ from monster to
monster: walk, fly, swim, climb, burrow, and hover. `actions` is an array of embedded documents. In a relational
design, the speeds become nullable columns or a separate table, and the actions become a separate table with a
foreign key. One document stores the monster's speeds and actions.

In [ ]:
from pprint import pprint

pprint(monsters.find_one(
    {"name": "Adult Red Dragon"},
    {"_id": 0, "name": 1, "type": 1, "challenge_rating": 1, "hit_points": 1,
     "speed": 1, "damage_immunities": 1, "actions.name": 1},
))

## 3. Query nested fields and arrays

Dot notation, such as `speed.fly`, refers to a field inside an embedded document
([query embedded documents](https://www.mongodb.com/docs/manual/tutorial/query-embedded-documents/)). The `$exists`
operator matches documents that have the field.

A condition on an array field matches a document when any element of the array meets the condition
([query an array](https://www.mongodb.com/docs/manual/tutorial/query-arrays/)). `damage_immunities` is an array of
strings, so `{"damage_immunities": "fire"}` matches every monster with fire in that list.

In [ ]:
total = monsters.count_documents({})
flyers = monsters.count_documents({"speed.fly": {"$exists": True}})
print(flyers, "of", total, "monsters have a flying speed")
print()

fire = list(monsters.find({"damage_immunities": "fire"}, {"_id": 0, "name": 1}))
print(len(fire), "monsters are immune to fire:")
print(", ".join(m["name"] for m in fire))

## 4. Compute a statistic with pandas

`find` returns a cursor: an object that produces the matching documents one at a time. `pd.DataFrame` reads the
cursor into a table with one row per document and one column per projected field.
[`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) then computes the number of
monsters and the median hit points for each creature type.

In [ ]:
import pandas as pd

df = pd.DataFrame(monsters.find(
    {}, {"_id": 0, "name": 1, "type": 1, "challenge_rating": 1, "hit_points": 1}
))

stats = (df.groupby("type")["hit_points"]
           .agg(["count", "median"])
           .sort_values("median", ascending=False))
stats

## 5. Save the result as a document

The statistic goes back into MongoDB as one new document in a second collection, `results`. `to_dict("records")`
turns the table into a list of dictionaries, one per row. The `rows` field stores that list as an array of
embedded documents. The `student` field holds `YOUR_NAME` from the form in Part 1.

On the shared class cluster, Gil reads the `results` collection directly. On your own cluster, the document also
appears in Atlas: click **Browse Collections** on your cluster, then database `tabletop`, then collection `results`.

In [ ]:
from datetime import datetime, timezone

doc = {
    "student": YOUR_NAME,
    "question": "Which types of monsters have the most hit points?",
    "computed_at": datetime.now(timezone.utc),
    "rows": stats.reset_index().to_dict("records"),
}
db["results"].insert_one(doc)

pprint(db["results"].find_one(
    {"student": YOUR_NAME},
    {"_id": 0, "rows": {"$slice": 3}},
    sort=[("computed_at", -1)],
))

## 6. An index on one field

`explain` is a command that reports the plan MongoDB uses to run a query. A query on a field with no index examines
every document in the collection, and `explain` reports the plan as a collection scan, `COLLSCAN`. An index on the
field stores the field's values in sorted order. Each value has a reference to its document. With the index, the
query examines only the documents that match, and `explain` reports the plan as an index scan, `IXSCAN`. On MongoDB
8.0 the index plan can appear as `EXPRESS_IXSCAN`, an index scan that skips the step where MongoDB compares
candidate plans.
([Indexes](https://www.mongodb.com/docs/manual/indexes/),
[explain results](https://www.mongodb.com/docs/manual/reference/explain-results/))

`plan_summary` runs `explain` and returns the plan stages, the number of documents examined, the number returned,
and the time in milliseconds. With 334 documents, both plans finish too fast for the time to differ. The number of
documents examined shows the difference: the collection scan examines all 334, and the index scan examines one. The
collection scan examines more documents as the collection grows; the index scan for one name still examines one.

[`create_index`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.create_index)
builds the index on `name`. When the index already exists, `create_index` leaves it as it is. A `hint` of
`{"$natural": 1}` makes the query skip every index and read the documents in the order they are stored, which gives
the plan without the index. The same query without the hint gives the plan with the index.

In [ ]:
def plan_summary(query, hint=None):
    find = {"find": "monsters", "filter": query}
    if hint:
        find["hint"] = hint
    exp = db.command("explain", find, verbosity="executionStats")
    plan = exp["queryPlanner"]["winningPlan"]
    plan = plan.get("queryPlan", plan)
    stages = []
    while plan:
        stages.append(plan["stage"])
        plan = plan.get("inputStage")
    s = exp["executionStats"]
    return {"stages": " <- ".join(stages),
            "docs_examined": s["totalDocsExamined"],
            "returned": s["nReturned"],
            "ms": s["executionTimeMillis"]}

query = {"name": "Goblin"}

monsters.create_index("name")

print("without the index:", plan_summary(query, hint={"$natural": 1}))
print("with the index:   ", plan_summary(query))

## 7. A vector search index on embeddings

An embedding is a list of numbers that a trained model computes from a piece of text. Texts with similar meanings have
lists of numbers that are close together. The model here,
[`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), computes 384 numbers for each
text.

`describe` builds one text per monster from its name, size, type, and the names and descriptions of its special
abilities and actions. The cell computes an embedding for each text and stores it in a new field, `embedding`, with
one [`bulk_write`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.bulk_write)
call. The first run downloads the model from Hugging Face (about 90 MB). Colab can print a warning about a missing
`HF_TOKEN` secret; the download works without one.

In [ ]:
from pymongo import UpdateOne
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

def describe(m):
    parts = [m["name"], m["size"], m["type"]]
    for a in m.get("special_abilities", []) + m.get("actions", []):
        parts.append(a["name"] + ": " + a.get("desc", ""))
    return ". ".join(parts)

docs = list(monsters.find({}, {"name": 1, "size": 1, "type": 1,
                               "special_abilities": 1, "actions": 1}))
vectors = model.encode([describe(m) for m in docs], normalize_embeddings=True)

monsters.bulk_write([
    UpdateOne({"_id": m["_id"]}, {"$set": {"embedding": v.tolist()}})
    for m, v in zip(docs, vectors)
])
print(len(vectors), "embeddings of", len(vectors[0]), "numbers each")

A [vector search index](https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-type/) on `embedding`
lets MongoDB find the documents whose embeddings are closest to a query embedding.
`numDimensions` is the length of each embedding, and `similarity` is the measure of closeness. Atlas builds the index
in the background. The cell checks every five seconds and stops when the index is queryable. Free clusters allow
three indexes in total, counting search indexes and vector search indexes together. The notebook creates one. When
the index already exists, from an earlier run or from another notebook on the same cluster, the cell uses it.

In [ ]:
import time
from pymongo.errors import OperationFailure
from pymongo.operations import SearchIndexModel

INDEX = "monster_vectors"

try:
    monsters.create_search_index(SearchIndexModel(
        name=INDEX,
        type="vectorSearch",
        definition={"fields": [{
            "type": "vector",
            "path": "embedding",
            "numDimensions": 384,
            "similarity": "cosine",
        }]},
    ))
    print("Created index", INDEX)
except OperationFailure:
    if not list(monsters.list_search_indexes(INDEX)):
        raise
    print("Index", INDEX, "already exists")

while True:
    info = list(monsters.list_search_indexes(INDEX))
    if info and info[0].get("queryable"):
        print("Index", INDEX, "is ready")
        break
    if info and info[0].get("status") == "FAILED":
        raise RuntimeError(info[0])
    print("building...")
    time.sleep(5)

[`$vectorSearch`](https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-stage/) is the first stage of
an aggregation pipeline: a list of stages that MongoDB runs in order, each stage working on the output of the stage
before it. The same model computes a query embedding from the query text. `$vectorSearch` returns up to `limit` documents
whose embeddings are closest to the query embedding. `numCandidates` is the number of candidate documents the
search compares before it returns the top `limit` documents. With cosine similarity, the score runs from 0 to 1. A
higher score means the document's embedding is closer to the query embedding. The second stage, `$project`, keeps the listed fields and adds the search score as
`score`.

In [ ]:
def search(text, k=5):
    query_vector = model.encode(text, normalize_embeddings=True).tolist()
    pipeline = [
        {"$vectorSearch": {
            "index": INDEX,
            "path": "embedding",
            "queryVector": query_vector,
            "numCandidates": 100,
            "limit": k,
        }},
        {"$project": {"_id": 0, "name": 1, "type": 1, "challenge_rating": 1,
                      "score": {"$meta": "vectorSearchScore"}}},
    ]
    return pd.DataFrame(monsters.aggregate(pipeline))

search("an undead spirit that drains the life of the living")

The five results are the monsters whose text from `describe` is closest in meaning to the query. `search` accepts any
text:

In [ ]:
search("a creature that lives underwater")